# ACS Model Variable Extraction

**Run this once.** Loads only the 9 columns needed by the MRP model from all four
ACS 5-year PUMS files (16 M rows), applies all recodes, filters to adults 18+,
and saves `data/acs_adults_model_vars.parquet`.

All other notebooks load from that parquet (seconds vs. minutes).

### Variables extracted (from model PDF)
| ACS column | Model variable | Notes |
|---|---|---|
| `PWGTP` | person weight | poststrat denominator |
| `STATE` | state FIPS | geographic intercept |
| `DIVISION` | region9 | Census division (1–9) |
| `PUMA` | puma_code | needed for county-level crosswalk |
| `SEX` → `gender` | Male / Female | individual-level RE |
| `RAC1P` + `HISP` → `race4` | White / Black / Hispanic / Other | individual-level RE |
| `SCHL` → `educ_category` | 1–4 | individual-level RE |
| `AGEP` | age filter | adults 18+ only |

In [1]:
import pandas as pd
import numpy as np

ACS_DIR  = "/Users/carmenk/Documents/CSS/Capstone/acs2024_csv_5year/"
OUT_PATH = "/Users/carmenk/Documents/CSS/Capstone/data/acs_adults_model_vars.parquet"

# Only the 9 columns the model needs
COLS = ["PWGTP", "STATE", "PUMA", "DIVISION", "SEX", "RAC1P", "HISP", "SCHL", "AGEP"]

parts = []
for suffix in ["a", "b", "c", "d"]:
    chunk = pd.read_csv(f"{ACS_DIR}psam_pus{suffix}.csv",
                        usecols=COLS, low_memory=False)
    parts.append(chunk)
    print(f"  psam_pus{suffix}.csv → {len(chunk):,} rows")

pums = pd.concat(parts, ignore_index=True)
print(f"\nTotal: {len(pums):,}")

  psam_pusa.csv → 4,748,656 rows
  psam_pusb.csv → 3,468,017 rows
  psam_pusc.csv → 3,757,739 rows
  psam_pusd.csv → 4,121,316 rows

Total: 16,095,728


In [2]:
# ── Recodes ───────────────────────────────────────────────────────────────────

# Adults only
pums = pums[pums["AGEP"] >= 18].copy()
print(f"Adults 18+: {len(pums):,}")

# gender
pums["gender"] = pums["SEX"].map({1: "Male", 2: "Female"})

# race4  — ethnicity (HISP) takes precedence over race (RAC1P)
hisp = pd.to_numeric(pums["HISP"], errors="coerce")
pums["race4"] = np.select(
    [hisp > 1,
     (hisp <= 1) & (pums["RAC1P"] == 1),
     (hisp <= 1) & (pums["RAC1P"] == 2)],
    ["Hispanic", "White", "Black"],
    default="Other"
)

# educ_category: (0,15]=1 LessHS | (15,17]=2 HS | (17,20]=3 SomeCol | (20,24]=4 BA+
pums["educ_category"] = pd.cut(
    pd.to_numeric(pums["SCHL"], errors="coerce"),
    bins=[0, 15, 17, 20, 24], labels=[1, 2, 3, 4]
).astype("Int8")

# region9 label from DIVISION
division_map = {
    1: "New England",    2: "Mid-Atlantic",      3: "E. North Central",
    4: "W. North Central", 5: "South Atlantic",  6: "E. South Central",
    7: "W. South Central", 8: "Mountain",        9: "Pacific",
}
pums["region9"] = pums["DIVISION"].map(division_map)

# zero-pad geographic keys for crosswalk
pums["state_fips"] = pums["STATE"].astype(str).str.zfill(2)
pums["puma_code"]  = pums["PUMA"].astype(str).str.zfill(5)

# Drop raw columns no longer needed
pums = pums.drop(columns=["SEX", "RAC1P", "HISP", "SCHL", "AGEP", "PUMA"])

print(f"\nFinal columns: {pums.columns.tolist()}")
print(pums.head(5).to_string())

Adults 18+: 13,044,338

Final columns: ['DIVISION', 'STATE', 'PWGTP', 'gender', 'race4', 'educ_category', 'region9', 'state_fips', 'puma_code']
   DIVISION  STATE  PWGTP  gender  race4  educ_category           region9 state_fips puma_code
0         6      1     11    Male  White              2  E. South Central         01     01301
1         6      1      9  Female  Black              3  E. South Central         01     01403
2         6      1      3  Female  White              2  E. South Central         01     01801
3         6      1     13  Female  Other              3  E. South Central         01     01501
4         6      1     41  Female  White              4  E. South Central         01     00700


In [3]:
# ── Save to Parquet (fast reload in future notebooks) ─────────────────────────
pums.to_parquet(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")
print(f"Rows: {len(pums):,} | Columns: {pums.shape[1]}")

# Verify round-trip
check = pd.read_parquet(OUT_PATH)
print(f"Verified read-back: {len(check):,} rows")

Saved → /Users/carmenk/Documents/CSS/Capstone/data/acs_adults_model_vars.parquet
Rows: 13,044,338 | Columns: 9
Verified read-back: 13,044,338 rows


## State-level ACS covariates: driving alone & same-sex households

Two additional predictors for the Howe (2015) MRP model:
- **`pct_drive_alone`**: share of workers 16+ who drove alone to work (`JWTRNS == 1`)
- **`pct_samesex_hh`**: share of housing-unit households with a same-sex spouse or partner (`RELSHIPP ∈ {22, 23}`)

Saved to `acs_state_extra_covariates.csv` in the processed test_data folder.

In [ ]:
COLS_EXTRA = ["SERIALNO", "SPORDER", "STATE", "PWGTP", "JWTRNS", "RELSHIPP"]

parts = []
for suffix in ["a", "b", "c", "d"]:
    chunk = pd.read_csv(f"{ACS_DIR}psam_pus{suffix}.csv",
                        usecols=COLS_EXTRA, low_memory=False)
    parts.append(chunk)
    print(f"  psam_pus{suffix}.csv → {len(chunk):,} rows")

raw = pd.concat(parts, ignore_index=True)
raw["SERIALNO"] = raw["SERIALNO"].astype(str)
raw["JWTRNS"]   = pd.to_numeric(raw["JWTRNS"],   errors="coerce")
raw["RELSHIPP"] = pd.to_numeric(raw["RELSHIPP"], errors="coerce")
print(f"\nTotal: {len(raw):,}")

In [ ]:
# ── DRIVE: % who drove alone among workers (JWTRNS not null) ──────────────────
workers = raw.dropna(subset=["JWTRNS"]).copy()
workers["drove_alone"] = (workers["JWTRNS"] == 1).astype(float)

drive = (
    workers.groupby("STATE")
    .apply(lambda g: np.average(g["drove_alone"], weights=g["PWGTP"]),
           include_groups=False)
    .reset_index(name="pct_drive_alone")
)

# ── SAME-SEX: % same-sex couple households (housing units only) ───────────────
# Housing units have "HU" in SERIALNO; group quarters have "GQ"
hu = raw[~raw["SERIALNO"].str.contains("GQ", na=False)].copy()

# Households with a same-sex spouse (22) or unmarried partner (23)
samesex_flag = (
    hu[hu["RELSHIPP"].isin([22, 23])][["SERIALNO"]]
    .drop_duplicates()
    .assign(has_samesex=1)
)

# Householder records carry the household-level weight (SPORDER == 1)
householders = hu[hu["SPORDER"] == 1][["SERIALNO", "STATE", "PWGTP"]].copy()
householders = householders.merge(samesex_flag, on="SERIALNO", how="left")
householders["has_samesex"] = householders["has_samesex"].fillna(0)

samesex = (
    householders.groupby("STATE")
    .apply(lambda g: np.average(g["has_samesex"], weights=g["PWGTP"]),
           include_groups=False)
    .reset_index(name="pct_samesex_hh")
)

# ── Merge and save ────────────────────────────────────────────────────────────
out = drive.merge(samesex, on="STATE")
out["state_fips"] = out["STATE"].astype(str).str.zfill(2)
out = out.drop(columns="STATE")

OUT_CSV = "/Users/carmenk/Documents/GitHub/MRdeeP-Deep-Learning-MRP/test_data/processed/acs_state_extra_covariates.csv"
out[["state_fips", "pct_drive_alone", "pct_samesex_hh"]].to_csv(OUT_CSV, index=False)

print(f"States: {len(out)}")
print(f"drive range:   [{out['pct_drive_alone'].min():.3f}, {out['pct_drive_alone'].max():.3f}]")
print(f"samesex range: [{out['pct_samesex_hh'].min():.4f}, {out['pct_samesex_hh'].max():.4f}]")
print(f"\nSaved → {OUT_CSV}")
print(out[["state_fips","pct_drive_alone","pct_samesex_hh"]].head(10).to_string(index=False))